In [1]:
from pathlib import Path
import json
import platform
import joblib
import numpy as np
import pandas as pd
import sklearn

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

PROJECT_DIR = Path.cwd()
MODELS_DIR = PROJECT_DIR / "models"
REPORTS_DIR = PROJECT_DIR / "reports"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.20

print("Project folder:", PROJECT_DIR)
print("Day 5 setup completed!")

Project folder: c:\Users\Work\Desktop\Testable ML training and inference pipeline\Day5-Capstone and reproducibility
Day 5 setup completed!


In [2]:
dataset = load_breast_cancer(as_frame=True)

X = dataset.data.copy()
y = dataset.target.copy()

FEATURE_COLUMNS = X.columns.tolist()

print("Dataset shape:", X.shape)
print("Target shape:", y.shape)
print("Features:", len(FEATURE_COLUMNS))
print("Target labels:", dataset.target_names)
print("\nTarget distribution:")
print(y.value_counts().sort_index())

Dataset shape: (569, 30)
Target shape: (569,)
Features: 30
Target labels: ['malignant' 'benign']

Target distribution:
target
0    212
1    357
Name: count, dtype: int64


In [3]:
assert isinstance(X, pd.DataFrame), "Features must be a DataFrame."
assert isinstance(y, pd.Series), "Target must be a Series."
assert not X.empty, "Dataset must not be empty."
assert len(X) == len(y), "Feature and target row counts differ."
assert not X.columns.duplicated().any(), "Duplicate feature columns found."
assert not y.isna().any(), "Target contains missing values."
assert y.nunique() >= 2, "Target must contain at least two classes."
assert all(
    pd.api.types.is_numeric_dtype(X[col])
    for col in X.columns
), "All features must be numeric."

assert not np.isinf(X.to_numpy(dtype=float)).any(), "Infinite values found."

quality_summary = {
    "rows": int(len(X)),
    "features": int(X.shape[1]),
    "missing_feature_values": int(X.isna().sum().sum()),
    "duplicate_rows": int(X.duplicated().sum()),
    "missing_targets": int(y.isna().sum())
}

print("Data validation: PASSED")
print(quality_summary)

Data validation: PASSED
{'rows': 569, 'features': 30, 'missing_feature_values': 0, 'duplicate_rows': 0, 'missing_targets': 0}


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print("Random state:", RANDOM_STATE)
print("Stratified split: enabled")

Training records: 455
Testing records: 114
Random state: 42
Stratified split: enabled


In [5]:
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=2000))
])

model.fit(X_train, y_train)

print("Model training completed successfully!")
print("Model:", type(model.named_steps["classifier"]).__name__)

Model training completed successfully!
Model: LogisticRegression


In [6]:
y_pred = model.predict(X_test)

metrics = {
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "precision_weighted": float(
        precision_score(y_test, y_pred, average="weighted", zero_division=0)
    ),
    "recall_weighted": float(
        recall_score(y_test, y_pred, average="weighted", zero_division=0)
    ),
    "f1_weighted": float(
        f1_score(y_test, y_pred, average="weighted", zero_division=0)
    ),
    "test_records": int(len(y_test)),
    "incorrect_predictions": int((y_test.to_numpy() != y_pred).sum())
}

print("Evaluation metrics:")
for metric_name, value in metrics.items():
    print(f"{metric_name}: {value}")

print("\nConfusion matrix (rows = actual, columns = predicted):")
print(confusion_matrix(y_test, y_pred, labels=[0, 1]))

print("\nClassification report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=dataset.target_names,
    zero_division=0
))

Evaluation metrics:
accuracy: 0.9824561403508771
precision_weighted: 0.9824561403508771
recall_weighted: 0.9824561403508771
f1_weighted: 0.9824561403508771
test_records: 114
incorrect_predictions: 2

Confusion matrix (rows = actual, columns = predicted):
[[41  1]
 [ 1 71]]

Classification report:
              precision    recall  f1-score   support

   malignant       0.98      0.98      0.98        42
      benign       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114



In [7]:
metadata = {
    "project": "Testable ML Training and Inference Pipeline",
    "day": 5,
    "dataset": "Breast Cancer Wisconsin (scikit-learn)",
    "model": "LogisticRegression",
    "total_records": int(len(X)),
    "feature_count": int(X.shape[1]),
    "feature_columns": FEATURE_COLUMNS,
    "target_column": "target",
    "target_labels": {
        "0": "malignant",
        "1": "benign"
    },
    "test_size": TEST_SIZE,
    "random_state": RANDOM_STATE,
    "stratified": True,
    "python_version": platform.python_version(),
    "pandas_version": pd.__version__,
    "scikit_learn_version": sklearn.__version__
}

bundle = {
    "model": model,
    "feature_columns": FEATURE_COLUMNS,
    "metadata": metadata
}

model_path = MODELS_DIR / "capstone_model.joblib"
metadata_path = MODELS_DIR / "capstone_metadata.json"

joblib.dump(bundle, model_path)

metadata_path.write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8"
)

print("Model saved:", model_path)
print("Metadata saved:", metadata_path)

Model saved: c:\Users\Work\Desktop\Testable ML training and inference pipeline\Day5-Capstone and reproducibility\models\capstone_model.joblib
Metadata saved: c:\Users\Work\Desktop\Testable ML training and inference pipeline\Day5-Capstone and reproducibility\models\capstone_metadata.json


In [8]:
metrics["confusion_matrix"] = confusion_matrix(
    y_test, y_pred, labels=[0, 1]
).tolist()

metrics_path = REPORTS_DIR / "capstone_metrics.json"

metrics_path.write_text(
    json.dumps(metrics, indent=2),
    encoding="utf-8"
)

print("Metrics saved:", metrics_path)
print(json.dumps(metrics, indent=2))

Metrics saved: c:\Users\Work\Desktop\Testable ML training and inference pipeline\Day5-Capstone and reproducibility\reports\capstone_metrics.json
{
  "accuracy": 0.9824561403508771,
  "precision_weighted": 0.9824561403508771,
  "recall_weighted": 0.9824561403508771,
  "f1_weighted": 0.9824561403508771,
  "test_records": 114,
  "incorrect_predictions": 2,
  "confusion_matrix": [
    [
      41,
      1
    ],
    [
      1,
      71
    ]
  ]
}


In [9]:
error_analysis = X_test.copy()
error_analysis["actual"] = y_test
error_analysis["predicted"] = y_pred

error_analysis = error_analysis[
    error_analysis["actual"] != error_analysis["predicted"]
].copy()

error_analysis["actual_label"] = error_analysis["actual"].map({
    0: "malignant",
    1: "benign"
})

error_analysis["predicted_label"] = error_analysis["predicted"].map({
    0: "malignant",
    1: "benign"
})

error_path = REPORTS_DIR / "capstone_error_analysis.csv"
error_analysis.to_csv(error_path, index=True)

print("Error analysis saved:", error_path)
print("Incorrect predictions recorded:", len(error_analysis))

Error analysis saved: c:\Users\Work\Desktop\Testable ML training and inference pipeline\Day5-Capstone and reproducibility\reports\capstone_error_analysis.csv
Incorrect predictions recorded: 2


In [10]:
# Negative test: reject data with a missing required feature

def validate_inference_input(data, expected_features):
    if not isinstance(data, pd.DataFrame):
        raise TypeError("Input must be a pandas DataFrame.")

    if data.empty:
        raise ValueError("Input must contain at least one row.")

    if data.columns.duplicated().any():
        raise ValueError("Duplicate feature columns are not allowed.")

    missing = set(expected_features) - set(data.columns)
    extra = set(data.columns) - set(expected_features)

    if missing:
        raise ValueError(f"Missing required features: {sorted(missing)}")

    if extra:
        raise ValueError(f"Unexpected features: {sorted(extra)}")

    if not all(
        pd.api.types.is_numeric_dtype(data[col])
        for col in expected_features
    ):
        raise TypeError("All features must be numeric.")

    return data[expected_features]


# A valid test row should pass
valid_row = X_test.iloc[[0]].copy()
validated_row = validate_inference_input(valid_row, FEATURE_COLUMNS)
print("Valid input test: PASSED")

# Remove one required feature; this should be rejected
invalid_row = valid_row.drop(columns=[FEATURE_COLUMNS[0]])

try:
    validate_inference_input(invalid_row, FEATURE_COLUMNS)
    raise AssertionError("Negative test failed: missing feature was accepted.")
except ValueError as exc:
    if "Missing required features" not in str(exc):
        raise
    print("Negative test: PASSED")
    print("Expected rejection:", exc)

Valid input test: PASSED
Negative test: PASSED
Expected rejection: Missing required features: ['mean radius']


In [11]:
report = f"""# Day 5 — Capstone and Reproducibility

## Project
Testable ML Training and Inference Pipeline

## Dataset and configuration
- Dataset: Breast Cancer Wisconsin (scikit-learn)
- Total records: {len(X)}
- Features: {X.shape[1]}
- Model: Logistic Regression
- Training records: {len(X_train)}
- Test records: {len(X_test)}
- Test size: {TEST_SIZE}
- Random state: {RANDOM_STATE}
- Stratified split: Yes

## Evaluation results
- Accuracy: {metrics['accuracy']:.4f}
- Weighted precision: {metrics['precision_weighted']:.4f}
- Weighted recall: {metrics['recall_weighted']:.4f}
- Weighted F1: {metrics['f1_weighted']:.4f}
- Incorrect predictions: {metrics['incorrect_predictions']}

## Error analysis
- Incorrect predictions saved: {len(error_analysis)}
- Error analysis file: reports/capstone_error_analysis.csv

## Validation evidence
- Dataset validation: passed
- Valid inference input: passed
- Missing-feature negative test: passed

## Reproducibility
- Random state: {RANDOM_STATE}
- Python version: {platform.python_version()}
- pandas version: {pd.__version__}
- scikit-learn version: {sklearn.__version__}

## How to run
Run all cells in Day5_Capstone_Reproducibility.ipynb from top to bottom.

## Limitations
- The dataset is public demonstration data, not company data.
- Results may differ on new data.
- The model is not intended for medical diagnosis or treatment decisions.
"""

report_path = REPORTS_DIR / "capstone_report.md"
report_path.write_text(report, encoding="utf-8")

print("Final report saved:", report_path)

Final report saved: c:\Users\Work\Desktop\Testable ML training and inference pipeline\Day5-Capstone and reproducibility\reports\capstone_report.md
